# Real Robot Practice

Hands-on exercises for the real UR3e robot. Run cells one by one.

**IPs:**
- UR3e1 (closest to window): `10.30.5.158`
- UR3e2: `10.30.5.159`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from URBasic import ISCoin, Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor
from math import radians

ROBOT_IP = "10.30.5.158"  # <-- change to your robot's IP
iscoin = ISCoin(host=ROBOT_IP, opened_gripper_size_mm=40)
rc = iscoin.robot_control  # shortcut

---
## 1. Read robot state
Find out where the robot is before doing anything.

In [ ]:
# Joint angles
joints = rc.get_actual_joint_positions()
print(f"Joint positions: {joints}")

# TCP (tool center point) in Cartesian space
tcp = rc.get_actual_tcp_pose()
print(f"TCP pose:        {tcp}")

# TCP force (useful to detect contact)
force = rc.force()
print(f"TCP force:       {force:.2f} N")

---
## 2. movej — Joint space moves
Move to known joint positions. The robot takes the fastest path in joint space (not necessarily a straight line in Cartesian space).

In [ ]:
# Reset any previous error
rc.reset_error()

# Save current position so we can return to it
start_joints = rc.get_actual_joint_positions()
print(f"Starting at: {start_joints}")

In [ ]:
# Move to home position
home = Joint6D.createFromRadians(1.1945, -1.1268, 1.0484, -1.5988, -1.5214, 1.0469)
rc.movej(home, a=radians(80), v=radians(60))
print(f"At home: {rc.get_actual_joint_positions()}")

In [ ]:
# Move to a different joint position
target = Joint6D.createFromRadians(1.7649, -1.1278, 1.0483, -1.5995, -1.4845, 1.0469)
rc.movej(target, a=radians(80), v=radians(60))
print(f"At target: {rc.get_actual_joint_positions()}")

In [ ]:
# Move back home
rc.movej(home, a=radians(80), v=radians(60))
print(f"Back home: {rc.get_actual_joint_positions()}")

---
## 3. movel — Linear Cartesian moves
Move in a straight line in Cartesian space. Useful for drawing, placing, or any task where the path matters.

In [ ]:
# Make sure we're at home first
rc.movej(home, a=radians(80), v=radians(60))

tcp_start = rc.get_actual_tcp_pose()
print(f"TCP start: {tcp_start}")

In [ ]:
# Move 5cm in X
target1 = TCP6D.createFromMetersRadians(
    tcp_start.x + 0.05, tcp_start.y, tcp_start.z,
    tcp_start.rx, tcp_start.ry, tcp_start.rz,
)
rc.movel(target1, v=0.1)
print(f"After +5cm X: {rc.get_actual_tcp_pose()}")

In [ ]:
# Move 5cm in Y
target2 = TCP6D.createFromMetersRadians(
    tcp_start.x + 0.05, tcp_start.y + 0.05, tcp_start.z,
    tcp_start.rx, tcp_start.ry, tcp_start.rz,
)
rc.movel(target2, v=0.1)
print(f"After +5cm Y: {rc.get_actual_tcp_pose()}")

In [ ]:
# Move back to start
rc.movel(tcp_start, v=0.1)
print(f"Back to start: {rc.get_actual_tcp_pose()}")

---
## 4. movel_waypoints — Continuous linear motion
Draw a 10cm square without stopping between corners. The blend radius `r` rounds the corners for smooth continuous motion.

In [ ]:
# Go home and read TCP
rc.movej(home, a=radians(80), v=radians(60))
tcp0 = rc.get_actual_tcp_pose()
print(f"TCP before square: {tcp0}")

In [ ]:
L = 0.10  # 10cm side length
v = 0.05  # slow speed for visibility
blend = 0.01  # 1cm blend radius at corners

corners = [
    TCP6D.createFromMetersRadians(tcp0.x + L, tcp0.y,     tcp0.z, tcp0.rx, tcp0.ry, tcp0.rz),
    TCP6D.createFromMetersRadians(tcp0.x + L, tcp0.y + L, tcp0.z, tcp0.rx, tcp0.ry, tcp0.rz),
    TCP6D.createFromMetersRadians(tcp0.x,     tcp0.y + L, tcp0.z, tcp0.rx, tcp0.ry, tcp0.rz),
    TCP6D.createFromMetersRadians(tcp0.x,     tcp0.y,     tcp0.z, tcp0.rx, tcp0.ry, tcp0.rz),
]

waypoints = [
    TCP6DDescriptor(corners[0], v=v, r=blend),
    TCP6DDescriptor(corners[1], v=v, r=blend),
    TCP6DDescriptor(corners[2], v=v, r=blend),
    TCP6DDescriptor(corners[3], v=v),  # last point: no blend, stop exactly
]

print("Drawing square with movel_waypoints...")
rc.movel_waypoints(waypoints)
print(f"TCP after square: {rc.get_actual_tcp_pose()}")

---
## 5. movec — Circular arc motion
Move along a circular arc defined by 3 points: current position, a via point, and the destination.

In [ ]:
# Go home
rc.movej(home, a=radians(80), v=radians(60))
tcp0 = rc.get_actual_tcp_pose()
print(f"TCP before arc: {tcp0}")

In [ ]:
# Draw a semicircle in XY plane
# Current position = start of arc
# via = top of the arc (offset in X and Y)
# to  = end of arc (offset in Y only, same X as start)

R = 0.05  # 5cm radius

pose_via = TCP6D.createFromMetersRadians(
    tcp0.x + R, tcp0.y + R, tcp0.z,
    tcp0.rx, tcp0.ry, tcp0.rz,
)

pose_to = TCP6D.createFromMetersRadians(
    tcp0.x, tcp0.y + 2 * R, tcp0.z,
    tcp0.rx, tcp0.ry, tcp0.rz,
)

print(f"Arc: start={tcp0}")
print(f"     via  ={pose_via}")
print(f"     to   ={pose_to}")

rc.movec(pose_via, pose_to, v=0.05)
print(f"\nTCP after arc: {rc.get_actual_tcp_pose()}")

In [ ]:
# Move back home to finish
rc.movel(tcp0, v=0.1)
print(f"Back to start: {rc.get_actual_tcp_pose()}")

---
## Cleanup

In [ ]:
iscoin.close()
print("Connection closed.")